In [21]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [22]:

# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [23]:
# Load the pre-trained model with ReLU activation
model = load_model('simplernn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [24]:
model.get_weights()

[array([[ 0.03896275,  0.01654013,  0.03358729, ...,  0.01386361,
          0.03708518, -0.010023  ],
        [-0.05937252,  0.01349449, -0.00617815, ..., -0.05220485,
         -0.02924802, -0.01042866],
        [ 0.02786779, -0.00903465, -0.03065408, ...,  0.00808512,
          0.00832796,  0.02675816],
        ...,
        [ 0.095999  , -0.11533975,  0.06277464, ...,  0.09306541,
         -0.08674704,  0.1450868 ],
        [ 0.04901781, -0.05412211, -0.0886101 , ..., -0.02415804,
         -0.04637798, -0.01666618],
        [ 0.1153779 , -0.01272244,  0.05537795, ...,  0.1542645 ,
         -0.10598161,  0.02047615]], shape=(10000, 128), dtype=float32),
 array([[ 0.0992671 ,  0.0015884 , -0.31216913, ...,  0.10900736,
          0.07154442, -0.09039869],
        [ 0.12385967,  0.11200113, -0.04503638, ...,  0.16550656,
         -0.09610499,  0.06849874],
        [ 0.16229346,  0.0925286 , -0.31500202, ...,  0.12398209,
          0.23383893,  0.11876241],
        ...,
        [ 0.1191234

In [25]:
# Step 2: Helper Functions
import re

# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    # Split into words and strip punctuation so "fantastic!" matches "fantastic"
    words = re.findall(r"\b\w+\b", text.lower())
    encoded_review = []
    for word in words:
        idx = word_index.get(word, -1)
        # Keep only the top 10,000 words the model was trained on;
        # everything else (unknown / out-of-vocab) maps to the OOV token 2
        if 0 <= idx < 9997:
            encoded_review.append(idx + 3)
        else:
            encoded_review.append(2)
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [26]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]



In [27]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.9260997176170349
